In [22]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/luisjaneirobezi/sssvsvsv/hotel_clean_dataset22.csv
/kaggle/input/datasets/luisjaneirobezi/hotel-bookings/hotel_bookings.csv


In [23]:
# ==============================
# 1. Importar librerías
# ==============================

import pandas as pd
import numpy as np

# ==============================
# 2. Cargar dataset
# ==============================

df = pd.read_csv("/kaggle/input/datasets/luisjaneirobezi/hotel-bookings/hotel_bookings.csv")

print("Filas iniciales:", len(df))


# ==============================
# 3. Reducir dataset a columnas necesarias
# ==============================

columns_needed = [
    "hotel",
    "market_segment",
    "adr",
    "stays_in_weekend_nights",
    "stays_in_week_nights"
]

df = df[columns_needed].copy()


# ==============================
# 4. Resolver problemas de formato
# ==============================

# Asegurar que ADR sea numérico (formato estándar con punto)
df["adr"] = (
    df["adr"]
    .astype(str)
    .str.replace(",", ".", regex=False)  # convertir coma a punto si existe
)

df["adr"] = pd.to_numeric(df["adr"], errors="coerce")

# Convertir noches a numérico
df["stays_in_weekend_nights"] = pd.to_numeric(
    df["stays_in_weekend_nights"], errors="coerce"
)

df["stays_in_week_nights"] = pd.to_numeric(
    df["stays_in_week_nights"], errors="coerce"
)


# ==============================
# 5. Limpiar categorías inconsistentes
# ==============================

df["market_segment"] = df["market_segment"].str.strip().str.lower()

df["market_segment"] = df["market_segment"].replace({
    "corporate": "Corporate",
    "direct": "Direct",
    "groups": "Groups",
    "offline ta/to": "Offline TA/TO",
    "online ta": "Online TA",
    "complementary": "Complementary",
    "aviation": "Aviation"
})

df["hotel"] = df["hotel"].str.strip()


# ==============================
# 6. Crear métricas base
# ==============================

df["room_nights"] = (
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)

df["revenue"] = (df["adr"] * df["room_nights"]).round(2)

# ==============================
# 7. Eliminar valores no válidos
# ==============================

rows_before = len(df)

df = df[
    (df["adr"] > 0) &
    (df["room_nights"] > 0)
]

rows_after = len(df)

rows_removed = rows_before - rows_after


# ==============================
# 8. Reset index
# ==============================

df = df.reset_index(drop=True)


# ==============================
# 9. Verificación final
# ==============================

print("\nColumnas finales:")
print(df.columns)

print("\nFilas eliminadas durante limpieza:", rows_removed)

print("\nPrimeras filas del dataset limpio:")
display(df.head())

Filas iniciales: 119390

Columnas finales:
Index(['hotel', 'market_segment', 'adr', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'room_nights', 'revenue'],
      dtype='object')

Filas eliminadas durante limpieza: 1960

Primeras filas del dataset limpio:


,hotel,market_segment,adr,stays_in_weekend_nights,stays_in_week_nights,room_nights,revenue
0,Resort Hotel,Direct,75.0,0,1,1,75.0
1,Resort Hotel,Corporate,75.0,0,1,1,75.0
2,Resort Hotel,Online TA,98.0,0,2,2,196.0
3,Resort Hotel,Online TA,98.0,0,2,2,196.0
4,Resort Hotel,Direct,107.0,0,2,2,214.0


In [24]:
# ==============================
# 10. Agrupar por segmento
# ==============================

segment_metrics = (
    df
    .groupby("market_segment")
    .agg(
        RoomNights=("room_nights", "sum"),
        Revenue=("revenue", "sum")
    )
)

# ==============================
# 11. Calcular ADR
# ==============================

segment_metrics["ADR"] = (
    segment_metrics["Revenue"] /
    segment_metrics["RoomNights"]
)

# ==============================
# 12. Totales del hotel
# ==============================

total_revenue = segment_metrics["Revenue"].sum()
total_roomnights = segment_metrics["RoomNights"].sum()

# ==============================
# 13. Revenue % y RoomNights %
# ==============================

segment_metrics["Revenue_%"] = (
    segment_metrics["Revenue"] /
    total_revenue
)

segment_metrics["RoomNights_%"] = (
    segment_metrics["RoomNights"] /
    total_roomnights
)

# ==============================
# 14. Segment Performance Index
# ==============================

segment_metrics["Segment_Performance_Index"] = (
    segment_metrics["Revenue_%"] /
    segment_metrics["RoomNights_%"]
)

# ==============================
# 15. Redondear para mejor lectura
# ==============================

segment_metrics = segment_metrics.round(3)

# ==============================
# 16. Mostrar resultado
# ==============================

display(segment_metrics.sort_values("Revenue", ascending=False))

,RoomNights,Revenue,ADR,Revenue_%,RoomNights_%,Segment_Performance_Index
market_segment,,,,,,
Online TA,201452,23942047.53,118.847,0.560,0.496,1.129
Offline TA/TO,94022,8151912.73,86.702,0.191,0.232,0.824
Direct,40056,5093028.39,127.148,0.119,0.099,1.208
Groups,58591,4669700.54,79.700,0.109,0.144,0.757
Corporate,10897,774295.26,71.056,0.018,0.027,0.675
Aviation,855,87446.36,102.276,0.002,0.002,0.972
Complementary,155,5082.52,32.790,0.000,0.000,0.312
undefined,3,48.00,16.000,0.000,0.000,0.152
